# 04 · Forecasting monthly deaths, tested on withheld 2026 data

### Protocol (identical for every fold)
Models are fitted **once** on data up to December *Y − 1* and then **frozen**. Two questions are scored on Jan–Sep of year *Y*:

* **Track A – operational, 1 month ahead.** Each month the frozen model receives the newly observed month (no
  refit) and forecasts the next one. This is how an early-warning cell would use it.
* **Track B – strategic, 1–9 months ahead** from a single origin (Dec *Y − 1*), using nothing from year *Y*.

**Back-test folds** *Y* = 2022, 2023, 2024, 2025 are used for **all** design choices: features, the ablation,
ensemble membership and interval calibration. **2026 (provisional UCDP Candidate)** is the withheld test set,
scored once at the end.

### Models
| Model | Why it is here |
|---|---|
| Naive, seasonal naive, 3-month moving average | Benchmarks any model must beat |
| ETS (damped Holt, log) | Strong classical smoother for level shifts |
| SARIMA (AIC-selected per series, log) | Classical autocorrelation and seasonality |
| **LightGBM-Δ** (global, all four conflicts) | Learns the *change* in log deaths from 18 dynamic features; the Δ-target removes the tree-ceiling problem |
| LightGBM bottom-up (Tweedie, 222 sub-national series, summed) | Answers *where*, and gives a hierarchical forecast |
| **Ensemble** | Log-space mean of members chosen on the back-test |

Metrics: MAE, RMSE, MAPE (actual > 0), sMAPE, MASE, and **RMSLE**, which is our selection metric because the four
series differ by three orders of magnitude.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from src import config as C, viz

In [2]:
from src import models as M
cm = pd.read_parquet(C.PROCESSED / "case_month.parquet")
am = pd.read_parquet(C.PROCESSED / "admin1_month.parquet")
fc = M.run_all(cm, am)          # 5 folds x 7 models (+ ensemble), ~1 minute
print("Ensemble members chosen on 2022-25 back-test:", fc["members"])

Ensemble members chosen on 2022-25 back-test: ['ma3', 'ets', 'lgb_case']


## 1. Design choices justified by ablation (back-test folds only)

In [3]:
abl = M.ablation(cm)
abl

track,RMSLE Track A (1-step),RMSLE Track B (1-9 step)
variant,,
"A. Level target, L2, full features (default ML)",1.232,1.197
"B. Δ target, L2, full features",0.993,1.260
"C. Δ target, L2, full features, regularised",0.955,1.283
"D. Δ target, Huber, compact features",0.896,1.143
"E. Δ target, L1, compact features (selected)",0.902,1.128
"F. Tweedie on raw counts, full features",1.331,1.263


**Reading.** A plain LightGBM on the log *level* (A) is the **worst** ML option: trees cannot predict beyond
the range seen in training, so a model fitted before Feb 2022 can never forecast the invasion's death toll.
Predicting the **change** (B–E) fixes this. A compact feature set with robust loss (E) generalises best, because
four conflict series cannot support 40 features. Raw-count Tweedie (F) is poor at conflict level.

## 2. Ensemble selection (back-test only)

In [4]:
fc['ensemble_table'].head(10)

,members,k,A,B,avg
0,ma3 + ets + lgb_case,3,0.839,1.142,0.990
1,naive + ma3 + ets + lgb_case,4,0.831,1.151,0.991
2,ma3 + lgb_case,2,0.861,1.128,0.995
3,ets + lgb_case,2,0.848,1.142,0.995
4,naive + ma3 + ets + lgb_admin1,4,0.860,1.134,0.997
5,naive + ets + lgb_case,3,0.835,1.162,0.999
6,naive + ma3 + ets,3,0.831,1.168,1.000
7,ma3 + ets + lgb_case + lgb_admin1,4,0.891,1.115,1.003
8,naive + ma3 + lgb_case,3,0.852,1.160,1.006
9,naive + ma3 + lgb_case + lgb_admin1,4,0.891,1.123,1.007


## 3. Results

In [5]:
r = fc["rmsle"]
tab = r.assign(set=np.where(r.fold == 2026, "2026 holdout", "back-test")).groupby(["track", "model", "set"]).RMSLE.mean().unstack().round(3)
tab.sort_values(("2026 holdout") if False else "2026 holdout")

set               2026 holdout  back-test
track model                              
A     sarima             0.844      1.065
      ensemble           0.845      0.839
      ets                0.897      0.867
      lgb_case           0.908      0.902
      lgb_admin1         0.933      1.293
      naive              0.961      0.928
B     lgb_admin1         1.003      1.208
A     ma3                1.014      0.907
B     sarima             1.042      1.532
      ets                1.062      1.196
      ensemble           1.169      1.142
      lgb_case           1.249      1.128
      naive              1.372      1.311
      ma3                1.464      1.182
A     snaive             1.579      1.798
B     snaive             1.579      1.798

In [6]:
viz.fig_model_compare(fc['rmsle'])

In [7]:
sc = fc["scores"]
hold = sc[sc.fold == 2026]
for metric in ["MAE", "RMSE", "MAPE", "MASE"]:
    print(f"\n2026 holdout - {metric} (Track A, 1 month ahead)")
    display(hold[hold.track == "A"].pivot_table(index="model", columns="conflict_case", values=metric).round(1))


2026 holdout - MAE (Track A, 1 month ahead)


conflict_case,Afghanistan–Pakistan,Israel–Hamas,Russia–Ukraine,US–Iran
model,,,,
ensemble,122.600,17.300,563.000,444.600
ets,113.300,34.700,805.300,437.800
lgb_admin1,148.500,101.300,"2,352.300",420.200
lgb_case,115.300,30.500,"1,438.900",433.300
ma3,145.400,29.200,"1,165.700",683.800
naive,128.300,38.500,805.300,643.300
sarima,98.200,35.200,686.000,434.000
snaive,144.300,"1,398.800","1,938.400",493.800



2026 holdout - RMSE (Track A, 1 month ahead)


conflict_case,Afghanistan–Pakistan,Israel–Hamas,Russia–Ukraine,US–Iran
model,,,,
ensemble,147.200,24.400,812.600,949.500
ets,140.700,44.100,923.300,965.200
lgb_admin1,162.800,126.300,"2,762.800",925.600
lgb_case,144.100,39.400,"1,559.400",947.000
ma3,172.800,47.900,"1,343.100","1,080.600"
naive,169.100,47.800,923.300,"1,121.300"
sarima,126.300,45.000,863.700,954.900
snaive,171.300,"1,550.900","2,104.800",953.900



2026 holdout - MAPE (Track A, 1 month ahead)


conflict_case,Afghanistan–Pakistan,Israel–Hamas,Russia–Ukraine,US–Iran
model,,,,
ensemble,44.300,13.900,9.100,397.100
ets,40.500,27.500,12.000,236.400
lgb_admin1,59.500,85.600,30.400,258.900
lgb_case,37.200,23.000,19.700,291.800
ma3,59.500,20.600,17.800,"2,023.500"
naive,46.200,29.200,12.000,611.400
sarima,31.800,27.800,10.400,256.800
snaive,46.800,"1,068.900",28.700,"1,458.900"



2026 holdout - MASE (Track A, 1 month ahead)


conflict_case,Afghanistan–Pakistan,Israel–Hamas,Russia–Ukraine,US–Iran
model,,,,
ensemble,1.300,0.100,0.800,11.900
ets,1.200,0.100,1.200,11.700
lgb_admin1,1.600,0.400,3.500,11.300
lgb_case,1.200,0.100,2.100,11.600
ma3,1.600,0.100,1.700,18.300
naive,1.400,0.100,1.200,17.200
sarima,1.100,0.100,1.000,11.600
snaive,1.600,5.000,2.900,13.200


In [8]:
base = hold[(hold.track == "A") & (hold.model == "naive")].set_index("conflict_case").MAE
ens = hold[(hold.track == "A") & (hold.model == "ensemble")].set_index("conflict_case").MAE
(1 - ens / base).mul(100).round(1).rename("% MAE reduction vs naive, 2026, 1-month ahead").to_frame()

,"% MAE reduction vs naive, 2026, 1-month ahead"
conflict_case,
Afghanistan–Pakistan,4.400
Israel–Hamas,55.200
Russia–Ukraine,30.100
US–Iran,30.900


## 4. Forecast vs actual on the withheld months

In [9]:
for case in C.CASES:
    viz.fig_forecast_case(cm, fc["preds"], case, track="A", future=fc["future"]).show()

In [10]:
for case in C.CASES:
    viz.fig_forecast_case(cm, fc["preds"], case, track="B").show()

## 5. Prediction intervals and reliability

In [11]:
p = fc["preds"]; e = p[p.model == "ensemble"]
cov = e.groupby(["fold", "track"]).apply(lambda g: pd.Series({
    "coverage_80": ((g.actual >= g.pi80_lo) & (g.actual <= g.pi80_hi)).mean(),
    "coverage_95": ((g.actual >= g.pi95_lo) & (g.actual <= g.pi95_hi)).mean(), "n": len(g)}))
cov.round(2)

coverage_80  coverage_95      n
fold track                                 
2022 A            0.750        0.940 36.000
     B            0.670        0.890 36.000
2023 A            0.780        1.000 36.000
     B            0.920        1.000 36.000
2024 A            0.860        1.000 36.000
     B            0.830        1.000 36.000
2025 A            0.780        0.830 36.000
     B            0.810        0.940 36.000
2026 A            0.810        0.920 36.000
     B            0.860        0.970 36.000

In [12]:
# directional reliability: did the forecast get the direction of change right (vs last observed month)?
a = e[e.track == "A"].sort_values(["conflict_case", "target_month"]).copy()
last = cm.set_index(["conflict_case", "month"]).deaths
a["last"] = [last.get((c, t - pd.DateOffset(months=1)), np.nan) for c, t in zip(a.conflict_case, a.target_month)]
a["dir_ok"] = np.sign(a.pred - a["last"]) == np.sign(a.actual - a["last"])
a.groupby("fold").dir_ok.mean().round(2).rename("direction accuracy (Track A)").to_frame()

,direction accuracy (Track A)
fold,
2022,0.580
2023,0.670
2024,0.580
2025,0.580
2026,0.720


In [13]:
bias = e.groupby(["fold", "conflict_case"]).apply(lambda g: np.median(np.log1p(g.pred) - np.log1p(g.actual)))
bias.unstack().round(2)

conflict_case,Afghanistan–Pakistan,Israel–Hamas,Russia–Ukraine,US–Iran
fold,,,,
2022,-0.120,-0.540,-4.340,-0.090
2023,0.230,-0.540,0.160,0.020
2024,-0.080,0.510,-0.140,-0.620
2025,-0.080,-0.160,0.090,0.290
2026,-0.200,0.140,0.080,-1.090


**Reading.**
* The 80% conformal intervals cover about **80–86%** of 2026 outcomes (target 80%), so they are well calibrated.
  The 95% upper tail is very wide by design: the back-test includes the 2022 invasion and the 2026 Gulf war, and
  that **outbreak risk** is real.
* Error is largest for US–Iran, whose 2026 war began with no build-up in the data. No model trained on monthly death
  counts can anticipate a decision to go to war, which is why we pair the forecast with sub-national escalation
  warning (notebook 05) and state this limit plainly.
* 2026 Candidate data are provisional and are usually revised **up**; the holdout error includes that revision noise.

## 6. What drives the forecast (TreeSHAP)

In [14]:
from src.features import make_supervised
sup = M._prep_case(make_supervised(cm, ["conflict_case"], horizons=range(1, 10)))
X = sup[(sup.month >= "2015-01-01") & sup.target_y.notna()][M.CASE_FEATURES + ["case_cat"]]
imp = M.shap_importance(fc["models_by_fold"][2026][0], X)
display(imp.head(12))
viz.fig_shap(imp)

,feature,mean_abs_shap
0,accel,0.239
1,y_l0,0.237
2,y_l2,0.150
3,momentum_3v12,0.125
4,y_rstd6,0.048
5,y_max12,0.045
6,y_rm12,0.042
7,tm_sin,0.040
8,y_rm6,0.033
9,y_ewm,0.031


## 7. Forward forecast, Oct–Dec 2026 (all data, refitted)

In [15]:
f = fc["future"]
f[f.model == "ensemble"][["conflict_case", "target_month", "pred", "pi80_lo", "pi80_hi"]].round(0)

,conflict_case,target_month,pred,pi80_lo,pi80_hi
84,Afghanistan–Pakistan,2026-10-01,318.000,155.000,786.000
85,Afghanistan–Pakistan,2026-11-01,316.000,143.000,"1,428.000"
86,Afghanistan–Pakistan,2026-12-01,291.000,132.000,"1,313.000"
87,Israel–Hamas,2026-10-01,128.000,62.000,317.000
88,Israel–Hamas,2026-11-01,129.000,58.000,583.000
89,Israel–Hamas,2026-12-01,123.000,55.000,557.000
90,Russia–Ukraine,2026-10-01,"6,780.000","3,316.000","16,708.000"
91,Russia–Ukraine,2026-11-01,"6,765.000","3,071.000","30,447.000"
92,Russia–Ukraine,2026-12-01,"6,735.000","3,058.000","30,316.000"
93,US–Iran,2026-10-01,26.000,12.000,66.000
